In [1]:
%pip install geopandas -q


StatementMeta(, ea6156ba-b99a-4e8c-ab5c-e39ede2a86b9, 7, Finished, Available, Finished, True)


Note: you may need to restart the kernel to use updated packages.



In [2]:
import geopandas as gpd, pandas as pd, numpy as np, os
from scipy.spatial import cKDTree

# ---------- Paramètres ----------
BRONZE = "/lakehouse/default/Files"
notebookutils.fs.mount("abfss://citiz_hackathon@onelake.dfs.fabric.microsoft.com/lakehouse_silver.Lakehouse", "/silver")
S = notebookutils.fs.getMountPath("/silver") + "/Files"
os.makedirs(f"{S}/features", exist_ok=True)

car = gpd.read_parquet(f"{S}/carreaux/carreaux_toulon_marseille.parquet").to_crs(2154).reset_index(drop=True)
opt = gpd.read_parquet(f"{S}/acces_5min/options.parquet")
acc = pd.read_parquet(f"{S}/acces_5min/acces_5min.parquet").merge(
    pd.DataFrame(opt[["option_id", "departs_heure_pointe", "capacite"]]), on="option_id")

# ---------- 1. Indicateurs population (déjà dans silver) ----------
f = car[["idcar_200m", "ville", "ind", "men", "densite_hab_km2", "part_sans_voiture", "menages_sans_voiture_est",
         "revenu_moyen", "part_65_plus", "geometry"]].copy()
f["part_logements_collectifs"] = (car["men_coll"] / car["men"]).clip(0, 1)

# ---------- 2. Transports à 5 min ----------
TC = ["bus", "tram", "metro", "train", "bateau", "telepherique"]
tc = acc[acc["type"].isin(TC)].groupby("idcar_200m").agg(nb_options_tc_5min=("option_id", "nunique"),
                                                        departs_pointe_5min=("departs_heure_pointe", "sum"))
velo  = acc[acc["type"] == "velo_station"].groupby("idcar_200m")["option_id"].nunique().rename("nb_velo_5min")
citiz = acc[acc["type"] == "citiz"].groupby("idcar_200m").agg(citiz_stations_5min=("option_id", "nunique"),
                                                             citiz_voitures_5min=("capacite", "sum"))
f = f.merge(tc, on="idcar_200m", how="left").merge(velo, on="idcar_200m", how="left").merge(citiz, on="idcar_200m", how="left")
for c in ["nb_options_tc_5min", "departs_pointe_5min", "nb_velo_5min", "citiz_stations_5min", "citiz_voitures_5min"]:
    f[c] = f[c].fillna(0)

# ---------- 3. OSM : activités et parkings à moins de 400 m ----------
centres = np.c_[f.centroid.x, f.centroid.y]
for ville in ["toulon", "marseille"]:
    m = (f.ville == ville).values
    for col, fichier in [("nb_activites_400m", "activites"), ("nb_parkings_400m", "parkings")]:
        p = gpd.read_parquet(f"{BRONZE}/osm/{ville}/{fichier}.parquet")
        p = p[p.geometry.notna()].to_crs(2154).representative_point()
        arbre = cKDTree(np.c_[p.x, p.y])
        f.loc[m, col] = [len(x) for x in arbre.query_ball_point(centres[m], 400)]

# ---------- 4. Distance à la gare la plus proche ----------
gares = opt[opt["type"] == "train"]
f["dist_gare_m"], _ = cKDTree(gares[["x", "y"]].values).query(centres)

# ---------- 5. Sauvegarder ----------
f.to_parquet(f"{S}/features/features_carreaux.parquet")
t = pd.DataFrame(f.drop(columns="geometry"))
for col in t.columns:
    if t[col].dtype == object: t[col] = t[col].astype(str)
sdf = spark.createDataFrame(t)
try:    sdf.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("lakehouse_silver.dbo.features_carreaux")
except Exception: sdf.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("lakehouse_silver.features_carreaux")
print("Sauvegardé : table 'features_carreaux' dans lakehouse_silver")

StatementMeta(, ea6156ba-b99a-4e8c-ab5c-e39ede2a86b9, 9, Finished, Available, Finished, False)

Sauvegardé : table 'features_carreaux' dans lakehouse_silver


In [3]:
cols = ["densite_hab_km2", "part_sans_voiture", "revenu_moyen", "part_logements_collectifs", "nb_options_tc_5min",
        "departs_pointe_5min", "nb_activites_400m", "nb_parkings_400m", "dist_gare_m"]
mrs = f[f.ville == "marseille"]
print("=== Marseille : carreaux AVEC Citiz à 5 min vs SANS (moyennes) ===")
print(pd.DataFrame({"avec Citiz": mrs[mrs.citiz_stations_5min > 0][cols].mean(),
                    "sans Citiz": mrs[mrs.citiz_stations_5min == 0][cols].mean()}).round(2))
print("\nCarreaux Marseille avec Citiz à 5 min :", int((mrs.citiz_stations_5min > 0).sum()), "sur", len(mrs))
print("\n=== Toulon (moyennes) ===")
print(f[f.ville == "toulon"][cols].mean().round(2))

StatementMeta(, ea6156ba-b99a-4e8c-ab5c-e39ede2a86b9, 10, Finished, Available, Finished, False)

=== Marseille : carreaux AVEC Citiz à 5 min vs SANS (moyennes) ===
                           avec Citiz  sans Citiz
densite_hab_km2              16873.76     5197.06
part_sans_voiture                0.42        0.20
revenu_moyen                 24700.88    26620.27
part_logements_collectifs        0.94        0.53
nb_options_tc_5min               4.72        1.58
departs_pointe_5min             66.86       23.45
nb_activites_400m               92.58       14.31
nb_parkings_400m                 9.31        7.25
dist_gare_m                   1625.74     2588.28

Carreaux Marseille avec Citiz à 5 min : 252 sur 3148

=== Toulon (moyennes) ===
densite_hab_km2               2261.25
part_sans_voiture                0.11
revenu_moyen                 27666.43
part_logements_collectifs        0.39
nb_options_tc_5min               1.05
departs_pointe_5min              8.95
nb_activites_400m                8.34
nb_parkings_400m                 5.17
dist_gare_m                   2940.48
dtype: flo